# Notebook 3 - Sentiment analysis

### By: FNF - Fight Fake News

## Environment Setup 

We would like to start with initialising Spark environment by specifying the full path to the Spark installation,enabling us to use PySpark in our notebook. Additionally, we added the exception handling with try/except block to ensure that any errors during initialization are caught and handled in timely manner, as the path might differ depending on the user's environment.

In [1]:
# Importing the findspark module 
import findspark

try:
    # Initializing via the full spark path
    findspark.init("/usr/local/spark/")
    
except Exception as e:
    print(f"Could not initialize Spark environment: {e}")
    print("Please check your Spark installation path.")

As the next step, we initialize a local PySpark environment with a SparkSession named "SentimentAnalysis" and allocate 2 GB executor memory. As well, we prepare both SparkContext and SQLContext for distributed data processing.

In [2]:
# Importing the SparkSession and SQLContent modules
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext

#Next, we are building the spark session, also including the possible exception handling 
try:
    spark = SparkSession.builder \
   .master("local[*]") \
   .appName("SentimentAnalysis") \
   .config("spark.executor.memory", "2g") \
   .getOrCreate()
except Exception as e:
    print("Spark session could not be created:", e)
    raise

# Geting the underlying SparkContext from the SparkSession
sc = spark.sparkContext

# Initializing SQLContext for working with structured data using SQL-like queries for our analysis and preprocessing
sqlContext = SQLContext(sc)

## Setup: Packages, Functions, and Modules

Now, we will import all, used in the context of this notebook 2, modules and functions.

In [3]:
# Importing the os module 
import os

# Set the PYSPARK_SUBMIT_ARGS to the appropriate spark-sql-kafka package
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.0.1 pyspark-shell'

In [4]:
# Installing textblob for the sentiment analysis
import sys
!{sys.executable} -m pip install -U textblob --no-cache-dir

Requirement already up-to-date: textblob in /opt/conda/lib/python3.8/site-packages (0.18.0.post0)


In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from textblob import TextBlob

from pyspark.ml.feature import Tokenizer #tokenizer
from pyspark.ml.feature import HashingTF, IDF # vectorizer
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

## Data Loading

After finishing up with all the setups, we start with loading the preprocessed main ISOT dataset as well as evaluation dataset from the notebook 1.

In [6]:
# Loading the data, also including the possible exception handling with the reading path
try:
    #main
    processed_df = spark.read.csv("./data/processed_df.csv", header=True, inferSchema=True, multiLine=True, escape='"', quote='"')
    #evalaution
    processed_eval_df = spark.read.csv("./data/processed_eval_df.csv", header=True, inferSchema=True, multiLine=True, escape='"', quote='"')

except Exception as e:
    print("Error loading Fake.csv:", e)

Now, we will briefly have a look at the data to see if the processed in the Notebook 1 datasets loaded as expected.

In [7]:
processed_df.show(5, truncate = 50)

+--------------------------------------------------+-----+
|                                              text|label|
+--------------------------------------------------+-----+
|president month leave office stage presidency w...|  1.0|
|year old german man admit court monday sell wea...|  0.0|
|split vote iowa caucus race hillary clinton ber...|  1.0|
|fbi director james comey defend handle democrat...|  0.0|
|top democrat house representative say friday co...|  0.0|
+--------------------------------------------------+-----+
only showing top 5 rows



In [8]:
processed_eval_df.show(5, truncate = 50)

+--------------------------------------------------+-----+
|                                              text|label|
+--------------------------------------------------+-----+
|white house say tuesday bill senate pass allow ...|  0.0|
|lithuania government lose majority parliament s...|  0.0|
|charlotte line wave ballot last month force vot...|  0.0|
|attorney represent illegal alien arrest allege ...|  1.0|
|trump administration outline tuesday dismantle ...|  0.0|
+--------------------------------------------------+-----+
only showing top 5 rows



Now, we will start with sentiment analysis.

# Sentiment Analysis

Now, with the use of TextBlob we can calculate the sentiment polarity and subjectivity of each news article, which measure how positive/negative and how objective/subjective the text is.

In [9]:
# text classification

# Define methods from TextBlob
def polarity_detection(text):
    return TextBlob(text).sentiment.polarity

def subjectivity_detection(text):
    return TextBlob(text).sentiment.subjectivity

# We need to create user defined functions for the Textblob methods in order to use them
def text_classification(words):
    # polarity detection
    # Define as user defined fuction to embed method in the spark environment 
    polarity_detection_udf = udf(polarity_detection, StringType())
    # Appending polarity to dataframe
    words = words.withColumn("polarity", polarity_detection_udf("text"))
    
    # subjectivity detection
    # Defining as user defined fuction to embed method in the spark environment 
    subjectivity_detection_udf = udf(subjectivity_detection, StringType())
    # Append subjectivity to dataframe 
    words = words.withColumn("subjectivity", subjectivity_detection_udf("text"))
    return words

As the next step, we add sentiment scores (polarity and subjectivity) to the data and removes rows without text, creating clean sentiment DataFrames for analysis.

In [10]:
sentiment_df = text_classification(processed_df).filter(col('text').isNotNull()) #main
sentiment_df_test = text_classification(processed_eval_df).filter(col('text').isNotNull()) #evaluation

Now, we are going to quickly inspect the data.

In [11]:
sentiment_df_test.show(5, truncate = 50)

+--------------------------------------------------+-----+---------------------+-------------------+
|                                              text|label|             polarity|       subjectivity|
+--------------------------------------------------+-----+---------------------+-------------------+
|white house say tuesday bill senate pass allow ...|  0.0| -0.16666666666666666| 0.3333333333333333|
|lithuania government lose majority parliament s...|  0.0|-0.035488505747126434|0.27902298850574714|
|charlotte line wave ballot last month force vot...|  0.0| 0.009648569023569035|  0.405370670995671|
|attorney represent illegal alien arrest allege ...|  1.0| -0.08849596349596349| 0.3706397956397956|
|trump administration outline tuesday dismantle ...|  0.0| 0.021428571428571432|0.30476190476190473|
+--------------------------------------------------+-----+---------------------+-------------------+
only showing top 5 rows



### Exploring Results

As the next step in our analysis we want to find answers to the following 2 questions:

1. Are fake news more negative or positive than real news on average?
2. Are fake news more subjective than real news?

In [12]:
sentiment_df.groupBy('label').agg(
    avg('polarity').alias('avg_polarity'),
    avg('subjectivity').alias('avg_subjectivity')).show()

+-----+--------------------+-------------------+
|label|        avg_polarity|   avg_subjectivity|
+-----+--------------------+-------------------+
|  0.0|0.050952648067841215|0.34262938633877066|
|  1.0| 0.05566383231458098| 0.4352070367074682|
+-----+--------------------+-------------------+



We can conclude that based on the average values fake news (label 1) seem to have a little bit more polarity to the positive side and seem to be significantly more subjective than real news (label 0).

## Predicting Real or Fake news based on sentiment

### Preparing features

First, we converts the sentiment scores for polarity and subjectivity into numeric values so they can be used as input features in the machine learning model, for both evaluation and main data frames.

In [13]:
# casting polarity and subjectivity values to numeric, so vector assembler can work with it
sentiment_df = sentiment_df.withColumn("polarity", col("polarity").cast(DoubleType()))
sentiment_df = sentiment_df.withColumn("subjectivity", col("subjectivity").cast(DoubleType()))

In [14]:
# now for the evaluation data frame
sentiment_df_test = sentiment_df_test.withColumn("polarity", col("polarity").cast(DoubleType()))
sentiment_df_test = sentiment_df_test.withColumn("subjectivity", col("subjectivity").cast(DoubleType()))

### Full Model with both sentiment and text

As the next step, we create a pipeline that sequentially applies a tokenizer, HashingTF, IDF, then combines the resulting text features with sentiment features (polarity and subjectivity) using a vector assembler, and finally applies logistic regression for classification as our best model as identified in notebook 2.

In [15]:
tokenizer = Tokenizer(inputCol="text", outputCol="words")

hashingTF = HashingTF(inputCol="words", outputCol="raw_features")

idf = IDF(inputCol="raw_features", outputCol="text_features")

assembler = VectorAssembler(
    inputCols=["text_features", "polarity", "subjectivity"],
    outputCol="features"
)

#Logistic regression  
lr = LogisticRegression(featuresCol="features", labelCol="label")


In [16]:
pipeline = Pipeline(stages=[tokenizer, hashingTF, idf, assembler, lr])

Now, we split into train/test dataframes, and assign sentiment_df_test to the new variable called diff_test_df to avoid confusion latter on.

In [17]:
train_df, test_df = sentiment_df.randomSplit([0.8, 0.2], seed=42)

diff_test_df = sentiment_df_test

Finally, we use the pipeline to train the model on the training data and then tested it on the test data. 

In [18]:
model = pipeline.fit(train_df)

predictions = model.transform(test_df)

Now, we will have a look at confusion matrix as well as recall, accuracy, precision scores.

In [19]:
confusion = predictions.groupBy("label", "prediction").count()
confusion.show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0| 4395|
|  0.0|       1.0|  237|
|  1.0|       0.0|  148|
|  0.0|       0.0| 4000|
+-----+----------+-----+



This confusion matrix shows the performance of our logistic regression model enhanced with sentiment features (polarity and subjectivity). The model correctly classified 4395 fake news articles as fake (true positives) and 4000 real news articles as real (true negatives). However, it misclassified 237 real articles as fake (false positives) and 148 fake articles as real (false negatives).

In [20]:
acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
print("Accuracy:", acc.evaluate(predictions))

Accuracy: 0.9561503416856492


The logistic regression model with added sentiment features achieves 95.61% accuracy, showing strong performance. Including polarity and subjectivity likely enhanced its ability to distinguish fake from real news.

In [21]:
prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="precisionByLabel", metricLabel=1.0)
print("Precision:", prec.evaluate(predictions))

Precision: 0.9488341968911918


The model's precision of 94.88% means that when it predicts an article as fake news, it is correct nearly 95% of the time, indicating very few false positives.


In [22]:
recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="recallByLabel", metricLabel=1.0)
print("Recall:", recall.evaluate(predictions))

Recall: 0.9674224081003742


Finally, our model's recall of 96.74% means the model successfully identifies most fake news articles, missing very few, showing strong detection capability.

### Testing with data of a different dataset

Now, we will check the performance of our model on the evaluation dataset, loaded after preprocessing from notebook 1.

In [23]:
predictions_diff = model.transform(diff_test_df)

First of all, we will have a look atat confusion matrix as well as recall, accuracy, and precision scores.

In [24]:
confusion = predictions_diff.groupBy("label", "prediction").count()
confusion.show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|33384|
|  0.0|       1.0| 9045|
|  1.0|       0.0| 1690|
|  0.0|       0.0|25972|
+-----+----------+-----+



On the unseen evaluation dataset, our logistic regression model with sentiment features correctly identified 33,384 fake news articles as fake (true positives) and 25,972 real news articles as real (true negatives). However, it misclassified 9,045 real articles as fake (false positives) and missed 1,690 fake articles, predicting them as real (false negatives). These results suggest that while our model has a strong ability to detect fake news (high recall), it also produces a considerable number of false alarms (false positives), which lowers its precision on this more varied evaluation dataset.

In [25]:
print("Accuracy:", acc.evaluate(predictions_diff)) #accuracy

print("Precision:", prec.evaluate(predictions_diff)) #presicion for fake news (1)

print("Recall:", recall.evaluate(predictions_diff)) # recall for fake news(1)

Accuracy: 0.8468419625914883
Precision: 0.7868203351481298
Recall: 0.9518161601186064


We can observe that on the evaluation dataset our model achieved an accuracy of approximately 84.68%, meaning it correctly classified that percentage of news articles overall. Its precision is about 78.68%, indicating that when the model predicts an article as fake news, it is correct nearly 79% of the time. Additionally, the recall is very high at 95.18%, showing that the model successfully identifies most of the actual fake news articles, minimizing missed detections.

To conclude, on the evaluation dataset, our logistic regression model with sentiment features achieved an accuracy around 84.68%, which is slightly higher than the 84.14% and 83.69% reported for other logistic models from notebook 2, including the cross-validated one (83.69%). This means it correctly classified roughly 84 to 85 out of every 100 news articles as real or fake.

Regarding precision (how often fake predictions are correct), the our model with sentiment features achieved about 78.68% precision, which is close to the 77.84% from the baseline logistic model on evaluation dataset from notebook 2 and slightly below the 79.17% precision of the cross-validated model from notebook 2. This suggests a moderate number of false positives (real news incorrectly flagged as fake), but the cross-validated model slightly improved on this.

For recall (how many actual fake news articles are detected), our model performed very well with 95.18% recall, meaning it successfully identified the vast majority of fake news. This is comparable or slightly better than the baseline recall of 95.5% but noticeably higher than the cross-validated model's recall of 91.49%.

So, we can conclude that inclusion of sentiment features (polarity and subjectivity) resulted in only a marginal improvement in our model performance. This suggests that while sentiment may have some relationship with fake news detection, it does not add substantial predictive value beyond the main textual features in our dataset but this can be also due to the limited scope of our dataset as well.